In [5]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import Tensor
from jaxtyping import Float, Int

/Users/peterbull/peter-projects/theograd/python/.venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [3]:
type(super().__init__())

RuntimeError: super(): no arguments

In [ ]:

class RNNModel(nn.Module):
    # Define __init__ with vocab_size, n_embed, and hidden_size as arguments
    def __init__(self, vocab_size: int = 65, n_embed: int = 32, hidden_size: int = 64) -> None:
    # Call the parent class constructor
      super().__init__()    
      self.hidden_size = hidden_size

      self.embed = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embed) # (65, 32)
      self.linear1 = nn.Linear(n_embed + self.hidden_size, hidden_size) # (96{32 + 64}, 64) -> ()
    # Store the Tanh activation (NOTE: this is the class itself, not an instance — bug to fix: should be nn.Tanh())
      self.tan_h = nn.Tanh()
    # Create linear2: projects from hidden space to vocab_size to produce logits (one score per token)


    # Define forward() with idx (token indices) and optional targets
    
        # Unpack the shape of idx into B (batch size) and T (sequence length)

        # Initialize the hidden state h0 as zeros with shape (B, hidden_size)
        # This is the "memory" of the RNN at t=0 — before seeing any tokens, we assume a blank slate
        # It lives on the same device as idx so we don't get CPU/GPU mismatches

        # Embed all tokens at once using the embedding table
        # Each token ID becomes a vector, so idx goes from (B, T) → (B, T, n_embed)
        # We do this upfront for efficiency instead of embedding one token per loop step

        # Create an empty list to collect the logit output at each timestep

        # Loop over each timestep t from 0 to T-1
            
            # Slice out the embedding for timestep t
            # embed[:, t, :] grabs all batches at position t → shape (B, n_embed)

            # Concatenate x_t and the previous hidden state h along the last dimension
            # x_t is shape (B, n_embed), h is shape (B, hidden_size)
            # After cat: (B, n_embed + hidden_size)
            # This is the core RNN idea: the input and memory are merged into one vector
            # so the network can use both "what am I seeing now" and "what have I seen before"

            # Pass the concatenated vector through linear1, then apply Tanh activation
            # linear1 maps (B, n_embed + hidden_size) → (B, hidden_size)
            # Tanh squashes values to (-1, 1), keeping the hidden state bounded across many timesteps
            # The result becomes the new h — this is the updated memory of the RNN

            # Pass the new hidden state through linear2 to get logits for this timestep
            # linear2 maps (B, hidden_size) → (B, vocab_size)
            # Each value is a raw score for how likely each token is to come next

            # Append the logit tensor for this timestep to the logits list

        # Stack the list of T logit tensors into a single tensor along dimension 1
        # Goes from a list of T tensors of shape (B, vocab_size) → (B, T, vocab_size)
        # Now we have one prediction per timestep, for every item in the batch

        # Initialize loss as None (only computed if targets are provided)

        # If targets were passed in, compute the cross entropy loss
            # Unpack logits shape into B, T, C (C = vocab_size)
            # Reshape logits from (B, T, C) → (B*T, C) — cross_entropy expects (N, C)
            # Reshape targets from (B, T) → (B*T,) — cross_entropy expects (N,) for class indices
            # Call F.cross_entropy on the reshaped logits and targets

        # Return logits and loss


    # Define count_params: sums the total number of learnable parameters in the model
    # p.numel() gives the number of elements in each parameter tensor


In [ ]:
emb = nn.Embedding(12, 6)

In [18]:
emb.weight

Parameter containing:
tensor([[-1.8492,  1.2675,  0.2563, -0.9460,  0.7848,  0.0920],
        [-0.0818, -1.0774, -0.2088, -1.3393,  0.5603, -1.0046],
        [ 0.5415, -0.2087, -0.0382, -0.7791,  0.4648, -0.9635],
        [ 0.1542, -0.5020, -0.2576, -0.6441, -0.4336, -0.0814],
        [ 0.2329, -0.3254, -0.2853,  1.8956, -0.8655, -0.4744],
        [ 0.4514, -0.2588,  0.9821, -0.3059,  0.3622,  0.6789],
        [ 0.8972, -0.8743,  1.1534,  0.0347, -0.0036, -1.0634],
        [-1.9366,  1.1687, -0.8025,  0.2779,  0.6060, -1.5074],
        [ 0.4140,  0.1316, -0.0920,  0.2523, -0.4127,  1.4774],
        [ 0.7553, -1.2112,  1.3662,  1.0095,  1.4831, -0.7966],
        [-0.7072,  0.9461, -0.3551, -0.4610,  1.0876,  1.1871],
        [ 0.9561,  0.7716, -1.3717, -0.1690,  1.2293, -0.4548]],
       requires_grad=True)

In [17]:
emb(torch.tensor([[1,2,3], [1,2,3]]))

tensor([[[-0.0818, -1.0774, -0.2088, -1.3393,  0.5603, -1.0046],
         [ 0.5415, -0.2087, -0.0382, -0.7791,  0.4648, -0.9635],
         [ 0.1542, -0.5020, -0.2576, -0.6441, -0.4336, -0.0814]],

        [[-0.0818, -1.0774, -0.2088, -1.3393,  0.5603, -1.0046],
         [ 0.5415, -0.2087, -0.0382, -0.7791,  0.4648, -0.9635],
         [ 0.1542, -0.5020, -0.2576, -0.6441, -0.4336, -0.0814]]],
       grad_fn=<EmbeddingBackward0>)